### Prof. Pedram Jahangiry

You need to make a copy to your own Google drive if you want to edit the original notebook! Start by opening this notebook on Colab 👇

<a href="https://colab.research.google.com/github/PJalgotrader/Deep_Learning-USU/blob/main/Platforms%20and%20tools/PyCaret/PyCaret-timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 



![logo](https://upload.wikimedia.org/wikipedia/commons/4/44/Huntsman-Wordmark-with-USU-Blue.gif#center) 


## 🔗 Links

[![linkedin](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pedram-jahangiry-cfa-5778015a)

[![Youtube](https://img.shields.io/badge/youtube_channel-1DA1F2?style=for-the-badge&logo=youtube&logoColor=white&color=FF0000)](https://www.youtube.com/channel/UCNDElcuuyX-2pSatVBDpJJQ)

[![Twitter URL](https://img.shields.io/twitter/url/https/twitter.com/PedramJahangiry.svg?style=social&label=Follow%20%40PedramJahangiry)](https://twitter.com/PedramJahangiry)


---


# ⚠️ Important: How to install PyCaret (read this first)

**We are using PyCaret 3.5.0, installed from the `pycaret-core` package.** The older `pycaret==3.3.2` release only supports Python 3.9 to 3.11 and is no longer updated, so `pip install pycaret` does not work on Google Colab or on any current Python. `pycaret-core` is the maintained continuation of PyCaret 3: same `import pycaret`, same functions, and it runs on Python 3.13.

### On Google Colab
Run the install cell below **first, in a fresh runtime**, before importing PyCaret. It detects Colab and runs:
```
pip install pycaret-core xgboost lightgbm catboost "statsmodels<0.15"
```
This installs the PyCaret core library plus the three boosting libraries used by the `*_cds_dt` reduced-regression forecasters. `pycaret-core` already includes sktime, statsmodels and pmdarima for the classical forecasters. The `statsmodels<0.15` pin is needed by the ETS forecaster: statsmodels 0.15 changed an internal method and sktime's ETS wrapper breaks on it (`ETSResults.simulate() got an unexpected keyword argument 'random_state'`), which makes `ets` silently drop out of `compare_models()` and fail in `tune_model()`.

No runtime restart is needed after the install. PyCaret checks which optional libraries exist the first time it is imported, so anything you install *after* importing it stays invisible (`Estimator ... Not Available`). If that happens, use Runtime ▸ Disconnect and delete runtime, then run the notebook from the top.

After the install finishes, verify with the version cell below (you should see **3.5.0**).

### On your own machine (conda)
The install cell does nothing outside Colab. Instead, create the course's PyCaret environment once, from the root of your cloned copy of this repo:
```
conda env create -f environment.yml
conda activate dl_pycaret
python scripts/check_environment.py
```
[`environment.yml`](https://github.com/PJalgotrader/Deep_Learning-USU/blob/main/environment.yml) uses **Python 3.13** (the same as Colab) and covers every PyCaret notebook in this course. The check script should end with "Your course environment is ready." Then open this notebook in JupyterLab or VS Code and pick the `dl_pycaret` environment as the kernel. Details and troubleshooting are in the [PyCaret setup guide](https://github.com/PJalgotrader/Deep_Learning-USU/blob/main/Platforms%20and%20tools/PyCaret/README.md). For a walkthrough of PyCaret itself, see my PyCaret playlist: https://www.youtube.com/playlist?list=PL2GWo47BFyUOqCAj_16yeNspfeM0nfA6q


Source:
1. https://py-caret.readthedocs.io/en/latest/tutorials.html (PyCaret tutorials)
2. https://py-caret.readthedocs.io/en/latest/api/time_series.html (time series API reference)
3. https://www.sktime.net/en/stable/api_reference/forecasting.html (sktime forecasting API, used under the hood)
4. https://pypi.org/project/pycaret-core/3.5.0/ (pycaret-core 3.5.0 on PyPI, the package we install in this notebook)
5. https://github.com/sktime/pycaret/releases (PyCaret release notes)


* If you want to search for model source code on GitHub, try searching the name of the model (for example rf_cds_dt) here: https://github.com/sktime/pycaret 
* For example, you want to see how the bootstrapping, stationarity or cross validation is handled in Pycaret. 
* for time series bootstrapping from scratch, try: https://arch.readthedocs.io/en/latest/bootstrap/timeseries-bootstraps.html
* for time series cross validation, you can change the fold strategy in pycaret. More advance CV like combinatorial purged is not built-in in pycaret (yet!)

# Installation

Follow the instructions at the top of this notebook: on Google Colab run the next cell; on your own machine use the `dl_pycaret` conda environment, where the next cell does nothing.

In [1]:
# Google Colab only: installs PyCaret 3.5.0 (pycaret-core) plus the libraries this notebook needs.
# On your own machine this cell does nothing (use the dl_pycaret conda environment, see the note above).
# statsmodels is pinned below 0.15 because sktime's ETS forecaster breaks on 0.15 (see the note above).
# Run it FIRST in a fresh runtime, before importing pycaret.
import sys
if "google.colab" in sys.modules:
    !pip install -q pycaret-core xgboost lightgbm catboost "statsmodels<0.15"

In [2]:
# let's double ckeck the Pycaret version: 
from pycaret.utils import version
version()

'3.5.0'

# Importing Dataset

In [3]:
from pycaret.datasets import get_data
airline = get_data('airline')

Period
1949-01    112.0
1949-02    118.0
1949-03    132.0
1949-04    129.0
1949-05    121.0
Freq: M, Name: Number of airline passengers, dtype: float64

In [4]:
from pycaret.time_series import *
setup(data = airline,  fh = 12)

,Description,Value
0,session_id,6572
1,Target,Number of airline passengers
2,Approach,Univariate
3,Exogenous Variables,Not Present
4,Original data shape,"(144, 1)"
5,Transformed data shape,"(144, 1)"
6,Transformed train set shape,"(132, 1)"
7,Transformed test set shape,"(12, 1)"
8,Rows with missing values,0.0%
9,Fold Generator,ExpandingWindowSplitter


In [5]:
models()

,Name,Reference,Turbo
ID,,,
naive,Naive Forecaster,sktime.forecasting.naive._naive.NaiveForecaster,True
grand_means,Grand Means Forecaster,sktime.forecasting.naive._naive.NaiveForecaster,True
snaive,Seasonal Naive Forecaster,sktime.forecasting.naive._naive.NaiveForecaster,True
polytrend,Polynomial Trend Forecaster,sktime.forecasting.trend._polynomial_trend_for...,True
arima,ARIMA,sktime.forecasting.arima._pmdarima.ARIMA,True
auto_arima,Auto ARIMA,sktime.forecasting.arima._pmdarima.AutoARIMA,True
exp_smooth,Exponential Smoothing,sktime.forecasting.exp_smoothing.ExponentialSm...,True
ets,ETS,sktime.forecasting.ets.AutoETS,True
theta,Theta Forecaster,sktime.forecasting.theta.ThetaForecaster,True


In [6]:
dt = create_model('dt_cds_dt')
tuned_dt = tune_model(dt)

,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,1956-12,0.6394,0.6691,18.6716,21.9026,0.0513,0.0508,0.8438
1,1957-12,0.6657,0.7266,20.3535,24.6684,0.0576,0.0551,0.8406
2,1958-12,0.8114,0.9486,23.1850,30.8350,0.0505,0.0519,0.7873
Mean,NaT,0.7055,0.7814,20.7367,25.8020,0.0531,0.0526,0.8239
SD,NaT,0.0756,0.1205,1.8624,3.7337,0.0032,0.0018,0.0259


,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,1956-12,0.5023,0.5886,14.6692,19.2683,0.0365,0.0375,0.8791
1,1957-12,0.8721,0.8809,26.6641,29.9075,0.0751,0.0717,0.7657
2,1958-12,0.5722,0.6560,16.3487,21.3251,0.0358,0.0365,0.8983
Mean,NaT,0.6489,0.7085,19.2273,23.5003,0.0491,0.0485,0.8477
SD,NaT,0.1604,0.1250,5.3031,4.6077,0.0184,0.0164,0.0585


Fitting 3 folds for each of 10 candidates, totalling 30 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    5.2s finished


## Compare Models

This function trains and evaluates performance of all estimators available in the model library using cross validation. The output of this function is a score grid with average cross validated scores. Metrics evaluated during CV can be accessed using the get_metrics function. Custom metrics can be added or removed using add_metric and remove_metric function.

In [7]:
compare_models(n_select=3)

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
exp_smooth,Exponential Smoothing,0.5717,0.5998,16.7798,19.7983,0.0422,0.0427,0.8953,0.0367
ets,ETS,0.5930,0.6212,17.4161,20.5095,0.0440,0.0445,0.8882,0.0433
et_cds_dt,Extra Trees w/ Cond. Deseasonalize & Detrending,0.6673,0.7318,19.6635,24.1977,0.0487,0.0487,0.8451,0.1733
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,0.6813,0.7866,20.0334,25.9670,0.0491,0.0499,0.8113,0.0933
arima,ARIMA,0.6830,0.6735,20.0069,22.2199,0.0501,0.0507,0.8677,0.0867
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,0.7004,0.7702,20.6084,25.4401,0.0509,0.0514,0.8215,0.0900
ridge_cds_dt,Ridge w/ Cond. Deseasonalize & Detrending,0.7004,0.7703,20.6086,25.4405,0.0509,0.0514,0.8215,0.0933
en_cds_dt,Elastic Net w/ Cond. Deseasonalize & Detrending,0.7029,0.7732,20.6816,25.5362,0.0511,0.0516,0.8201,0.0933
lasso_cds_dt,Lasso w/ Cond. Deseasonalize & Detrending,0.7048,0.7751,20.7373,25.6005,0.0512,0.0517,0.8193,0.0900
llar_cds_dt,Lasso Least Angular Regressor w/ Cond. Deseasonalize & Detrending,0.7048,0.7751,20.7366,25.6009,0.0512,0.0517,0.8192,0.0933


[ExponentialSmoothing(seasonal='mul', sp=12, trend='add'),
 AutoETS(seasonal='mul', sp=12, trend='add'),
 BaseCdsDtForecaster(fe_target_rr=[WindowSummarizer(lag_feature={'lag': [np.int64(12),
                                                                         np.int64(11),
                                                                         np.int64(10),
                                                                         np.int64(9),
                                                                         np.int64(8),
                                                                         np.int64(7),
                                                                         np.int64(6),
                                                                         np.int64(5),
                                                                         np.int64(4),
                                                                         np.int64(3),
                                

## Plot Model

This function analyzes the performance of a trained model on holdout set. It may require re-training the model in certain cases

In [8]:
exp_smooth = create_model('exp_smooth')

,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,1956-12,0.4989,0.5738,14.5688,18.7829,0.0367,0.0377,0.8852
1,1957-12,0.5088,0.5368,15.5551,18.2245,0.0420,0.0411,0.9130
2,1958-12,0.7075,0.6887,20.2157,22.3876,0.0479,0.0494,0.8879
Mean,NaT,0.5717,0.5998,16.7798,19.7983,0.0422,0.0427,0.8953
SD,NaT,0.0961,0.0647,2.4626,1.8450,0.0046,0.0049,0.0125


In [9]:
ets = create_model('ets')

,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,1956-12,0.4985,0.5734,14.5568,18.7704,0.0366,0.0376,0.8853
1,1957-12,0.5495,0.5693,16.8010,19.3266,0.0458,0.0447,0.9021
2,1958-12,0.7311,0.7208,20.8906,23.4316,0.0495,0.0512,0.8772
Mean,NaT,0.5930,0.6212,17.4161,20.5095,0.0440,0.0445,0.8882
SD,NaT,0.0998,0.0705,2.6221,2.0786,0.0054,0.0055,0.0104


In [10]:
plot_model(ets)

In [11]:
plot_model(plot="diff", data_kwargs={"order_list": [1, 2], "acf": True, "pacf": True})


In [12]:
plot_model(plot="diff", data_kwargs={"lags_list": [[1], [1, 12]], "acf": True, "pacf": True})


In [13]:
plot_model(ets, plot = 'ts')


In [14]:
plot_model(plot = 'cv')


In [15]:
plot_model(plot = 'decomp')


In [16]:
plot_model(plot = 'decomp', data_kwargs = {'type' : 'multiplicative'})


In [17]:
arima = create_model("arima")

,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,1956-12,0.4462,0.4933,13.0286,16.1485,0.0327,0.0334,0.9151
1,1957-12,0.5983,0.5993,18.2920,20.3442,0.0506,0.0491,0.8916
2,1958-12,1.0044,0.9280,28.6999,30.1669,0.0671,0.0697,0.7964
Mean,NaT,0.6830,0.6735,20.0069,22.2199,0.0501,0.0507,0.8677
SD,NaT,0.2356,0.1851,6.5117,5.8746,0.0141,0.0148,0.0513


In [18]:
plot_model(estimator = arima, plot = 'forecast', data_kwargs = {'fh' : 36})


In [19]:
tuned_arima = tune_model(arima)


,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,1956-12,0.4167,0.4694,12.1687,15.3643,0.0304,0.0310,0.9232
1,1957-12,0.6193,0.6193,18.9323,21.0258,0.0524,0.0508,0.8842
2,1958-12,0.7405,0.7211,21.1605,23.4388,0.0490,0.0505,0.8771
Mean,NaT,0.5922,0.6033,17.4205,19.9429,0.0439,0.0441,0.8948
SD,NaT,0.1336,0.1034,3.8234,3.3842,0.0097,0.0092,0.0203


Fitting 3 folds for each of 10 candidates, totalling 30 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    0.7s finished


In [20]:
plot_model([arima, tuned_arima, ets, exp_smooth], data_kwargs={"labels": ["ARIMA", "Tuned ARIMA", "ETS", "EXP_Smooth"]})


In [21]:
plot_model(plot="acf")

In [22]:
plot_model(plot="decomp_stl")

In [23]:
plot_model(plot="diagnostics")

In [24]:
plot_model(plot="ccf")

In [25]:
plot_model(estimator=ets, plot="residuals")

## Blending and Stacking

* **Blend** This function trains a EnsembleForecaster for select models passed in the estimator_list param. The output of this function is a score grid with CV scores by fold. Metrics evaluated during CV can be accessed using the get_metrics function. Custom metrics can be added or removed using add_metric and remove_metric function.

In [26]:
top3 = compare_models(n_select = 3)
blender = blend_models(top3)

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
exp_smooth,Exponential Smoothing,0.5717,0.5998,16.7798,19.7983,0.0422,0.0427,0.8953,0.0267
ets,ETS,0.5930,0.6212,17.4161,20.5095,0.0440,0.0445,0.8882,0.0333
et_cds_dt,Extra Trees w/ Cond. Deseasonalize & Detrending,0.6673,0.7318,19.6635,24.1977,0.0487,0.0487,0.8451,0.1733
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,0.6813,0.7866,20.0334,25.9670,0.0491,0.0499,0.8113,0.0933
arima,ARIMA,0.6830,0.6735,20.0069,22.2199,0.0501,0.0507,0.8677,0.0233
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,0.7004,0.7702,20.6084,25.4401,0.0509,0.0514,0.8215,0.0900
ridge_cds_dt,Ridge w/ Cond. Deseasonalize & Detrending,0.7004,0.7703,20.6086,25.4405,0.0509,0.0514,0.8215,0.0900
en_cds_dt,Elastic Net w/ Cond. Deseasonalize & Detrending,0.7029,0.7732,20.6816,25.5362,0.0511,0.0516,0.8201,0.0933
lasso_cds_dt,Lasso w/ Cond. Deseasonalize & Detrending,0.7048,0.7751,20.7373,25.6005,0.0512,0.0517,0.8193,0.0900
llar_cds_dt,Lasso Least Angular Regressor w/ Cond. Deseasonalize & Detrending,0.7048,0.7751,20.7366,25.6009,0.0512,0.0517,0.8192,0.0933


,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,1956-12,0.4799,0.5577,14.0144,18.2551,0.0348,0.0356,0.8915
1,1957-12,0.5960,0.5925,18.2226,20.1144,0.0497,0.0484,0.8940
2,1958-12,0.6804,0.6711,19.4404,21.8133,0.0442,0.0455,0.8936
Mean,NaT,0.5854,0.6071,17.2258,20.0609,0.0429,0.0432,0.8930
SD,NaT,0.0822,0.0474,2.3246,1.4531,0.0062,0.0055,0.0011


In [27]:
blender

EnsembleForecaster(forecasters=[('Exponential Smoothing',
                                 ExponentialSmoothing(seasonal='mul', sp=12,
                                                      trend='add')),
                                ('ETS',
                                 AutoETS(seasonal='mul', sp=12, trend='add')),
                                ('ExtraTreesRegressor',
                                 BaseCdsDtForecaster(fe_target_rr=[WindowSummarizer(lag_feature={'lag': [np.int64(12),
                                                                                                         np.int64(11),
                                                                                                         np.int64(10),
                                                                                                         np.int64(9),
                                                                                                         np.int64(8),
                                                                                                         np.int64(7),
                                                                                                         np.int64(6),
                                                                                                         np.int64(5),
                                                                                                         np.int64(4),
                                                                                                         np.int64(3),
                                                                                                         np.int64(2),
                                                                                                         np.int64(1)]},
                                                                                    n_jobs=1)],
                                                     regressor=ExtraTreesRegressor(n_jobs=-1, random_state=6572),
                                                     sp=12,
                                                     window_length=12))],
                   n_jobs=-1)

## Predict Model

This function predicts Label using a trained model. When data is None, it predicts label on the holdout set.

note: so far, our best model is the stacker model with the highest CV R2. :)


In [28]:
holdout_pred = predict_model(exp_smooth)

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,Exponential Smoothing,0.3384,0.4576,10.3031,15.8104,0.0221,0.0216,0.9549


## Finalize Model

This function trains a given estimator on the entire dataset including the holdout set.

Model finalization is the last step in the experiment. A normal machine learning workflow in PyCaret starts with setup(), followed by comparing all models using compare_models() and shortlisting a few candidate models (based on the metric of interest) to perform several modeling techniques such as hyperparameter tuning, ensembling, stacking etc. This workflow will eventually lead you to the best model for use in making predictions on new and unseen data. The finalize_model() function fits the model onto the complete dataset including the test/hold-out sample (30% in this case). The purpose of this function is to train the model on the complete dataset before it is deployed in production.

In [29]:
final_blender = finalize_model(blender)

In [30]:
final_blender

ForecastingPipeline(steps=[('forecaster',
                            TransformedTargetForecaster(steps=[('model',
                                                                EnsembleForecaster(forecasters=[('Exponential '
                                                                                                 'Smoothing',
                                                                                                 ExponentialSmoothing(seasonal='mul',
                                                                                                                      sp=12,
                                                                                                                      trend='add')),
                                                                                                ('ETS',
                                                                                                 AutoETS(seasonal='mul',
                                                                                                         sp=12,
                                                                                                         trend='add')),
                                                                                                ('ExtraTreesRegressor',
                                                                                                 BaseCdsDtForecaster(fe_target_rr=[WindowSummarizer(lag_feature={'lag': [np.int64(12),
                                                                                                                                                                         np.int64(11),
                                                                                                                                                                         np.int64(10),
                                                                                                                                                                         np.int64(9),
                                                                                                                                                                         np.int64(8),
                                                                                                                                                                         np.int64(7),
                                                                                                                                                                         np.int64(6),
                                                                                                                                                                         np.int64(5),
                                                                                                                                                                         np.int64(4),
                                                                                                                                                                         np.int64(3),
                                                                                                                                                                         np.int64(2),
                                                                                                                                                                         np.int64(1)]},
                                                                                                                                                    n_jobs=1)],
                                                                                                                     regressor=ExtraTreesRegressor(n_jobs=-1, random_state=6572),
                                                                                                                     sp=12,
                                           

### Final prediciton on unseen data

The predict_model() function is also used to predict on the unseen dataset. The only difference from section 11 above is that this time we will pass the data_unseen parameter. data_unseen is the variable created at the beginning of the tutorial and contains 10% of the original dataset which was never exposed to PyCaret.

In [31]:
unseen_predictions = predict_model(finalize_model(arima), fh=24)
unseen_predictions.tail()

,y_pred
1962-08,667.5857
1962-09,569.6943
1962-10,522.7764
1962-11,451.8385
1962-12,493.8855


## Save Model

This function saves the transformation pipeline and trained model object into the current working directory as a pickle file for later use.

In [32]:
save_model(final_blender, 'my_pycaret_ts_regression')

Transformation Pipeline and Model Successfully Saved


(ForecastingPipeline(steps=[('forecaster',
                             TransformedTargetForecaster(steps=[('model',
                                                                 ForecastingPipeline(steps=[('forecaster',
                                                                                             TransformedTargetForecaster(steps=[('model',
                                                                                                                                 EnsembleForecaster(forecasters=[('Exponential '
                                                                                                                                                                  'Smoothing',
                                                                                                                                                                  ExponentialSmoothing(seasonal='mul',
                                                                                       

## Load model

This function loads a previously saved pipeline.



In [33]:
my_winning_regressor = load_model('my_pycaret_ts_regression')

Transformation Pipeline and Model Successfully Loaded


In [34]:
my_winning_regressor

ForecastingPipeline(steps=[('forecaster',
                            TransformedTargetForecaster(steps=[('model',
                                                                ForecastingPipeline(steps=[('forecaster',
                                                                                            TransformedTargetForecaster(steps=[('model',
                                                                                                                                EnsembleForecaster(forecasters=[('Exponential '
                                                                                                                                                                 'Smoothing',
                                                                                                                                                                 ExponentialSmoothing(seasonal='mul',
                                                                                                                                                                                      sp=12,
                                                                                                                                                                                      trend='add')),
                                                                                                                                                                ('ETS',
                                                                                                                                                                 AutoETS(seasonal='mul',
                                                                                                                                                                         sp=12,
                                                                                                                                                                         trend='add')),
                                                                                                                                                                ('ExtraTreesRegressor',
                                                                                                                                                                 BaseCdsDtForecaster(fe_target_rr=[WindowSummarizer(lag_feature={'lag': [np.int64(12),
                                                                                                                                                                                                                                         np.int64(11),
                                                                                                                                                                                                                                         np.int64(10),
                                                                                                                                                                                                                                         np.int64(9),
                                                                                                                                                                                                                                         np.int64(8),
                                                                                                                                                                                                                                         np.int64(7),
                                                                                                                                                                                                                                         np.int64(6),
                                                                                      

## Deploy Model

This function deploys the transformation pipeline and trained model on cloud.

https://py-caret.readthedocs.io/en/latest/api/generated/pycaret.regression.deploy_model.html